# 🚗 GetAround MODELES & MLFLOW

In [1]:
import mlflow
print(mlflow.__version__)


2.21.3


In [2]:
import sklearn
import xgboost
import pandas
import mlflow
import sys
print(f"Python version: {sys.version.split(' ')[0]}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"xgboost: {xgboost.__version__}")
print(f"pandas: {pandas.__version__}")
print(f"mlflow: {mlflow.__version__}")

Python version: 3.12.9
scikit-learn: 1.6.1
xgboost: 3.0.3
pandas: 2.2.3
mlflow: 2.21.3


## Data Cleaning

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import mlflow, os
#from huggingface_hub import notebook_login
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
WIDTH = 800
from dotenv import load_dotenv
load_dotenv()



True

In [4]:
price_df = pd.read_csv("get_around_pricing_project.csv")

df = price_df.copy()

In [5]:
import plotly.express as px
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 1. Nettoyage des données : supprimer les valeurs négatives (si elles sont erronées)
for col in ['mileage', 'engine_power', 'rental_price_per_day']:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # Convertir en numérique
    df = df[df[col] >= 0]  # Garder seulement les valeurs positives

# 2. Vérification rapide des statistiques après nettoyage
print("Statistiques après nettoyage :")
print(df[['mileage', 'engine_power', 'rental_price_per_day']].describe())

# 3. Création d'un boxplot par variable (plus clair)
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=("Mileage", "Engine Power", "Rental Price per Day"))

fig.add_trace(go.Box(y=df['mileage'], name='Mileage'), row=1, col=1)
fig.add_trace(go.Box(y=df['engine_power'], name='Engine Power'), row=1, col=2)
fig.add_trace(go.Box(y=df['rental_price_per_day'], name='Rental Price per Day'), row=1, col=3)

fig.update_layout(title_text="Boxplots des colonnes nettoyées",
                  showlegend=False,
                  height=500,
                  width=1000)

fig.show()



Statistiques après nettoyage :
            mileage  engine_power  rental_price_per_day
count  4.842000e+03   4842.000000           4842.000000
mean   1.409919e+05    128.967369            121.182982
std    6.016882e+04     38.970348             33.499826
min    4.760000e+02      0.000000             10.000000
25%    1.029658e+05    100.000000            104.000000
50%    1.410845e+05    120.000000            119.000000
75%    1.752062e+05    135.000000            136.000000
max    1.000376e+06    423.000000            422.000000


In [6]:
price_df = price_df[price_df['mileage'] > 0]
price_df 

,Unnamed: 0,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4838,4838,Toyota,39743,110,diesel,black,van,False,True,False,False,False,False,True,121
4839,4839,Toyota,49832,100,diesel,grey,van,False,True,False,False,False,False,True,132
4840,4840,Toyota,19633,110,diesel,grey,van,False,True,False,False,False,False,True,130
4841,4841,Toyota,27920,110,diesel,brown,van,True,True,False,False,False,False,True,151


In [7]:
#mask enlever la donnée aberrante engine_power = 0
price_df = price_df[price_df['engine_power'] != 0]

price_df 

,Unnamed: 0,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4838,4838,Toyota,39743,110,diesel,black,van,False,True,False,False,False,False,True,121
4839,4839,Toyota,49832,100,diesel,grey,van,False,True,False,False,False,False,True,132
4840,4840,Toyota,19633,110,diesel,grey,van,False,True,False,False,False,False,True,130
4841,4841,Toyota,27920,110,diesel,brown,van,True,True,False,False,False,False,True,151


In [8]:
from skimpy import skim
skim(price_df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 4841   │ │ bool        │ 7     │                                                          │
│ │ Number of columns │ 15     │ │ int64       │ 4     │                                                          │
│ └───────────────────┴────────┘ │ string      │ 4     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column                 ┃ NA  ┃ NA %  ┃ mean    ┃ sd    ┃ p0  ┃ p25    ┃ p50    ┃ p75    ┃ p100    ┃ hist   ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━┩  │
│ │ Unnamed: 0             │   0 │     0 │    2421 │  1398 │   0 │   1210 │   2420 │   3631 │    4842 │ ██████ │  │
│ │ mileage                │   0 │     0 │  141000 │ 60170 │ 476 │ 103000 │ 141100 │ 175200 │ 1000000 │   █▃   │  │
│ │ engine_power           │   0 │     0 │     129 │ 38.93 │  25 │    100 │    120 │    135 │     423 │  ▂█▁▁  │  │
│ │ rental_price_per_day   │   0 │     0 │   121.2 │  33.5 │  10 │    104 │    119 │    136 │     422 │  ▁█▂   │  │
│ └────────────────────────┴─────┴───────┴─────────┴───────┴─────┴────────┴────────┴────────┴─────────┴────────┘  │
│                                                      bool                                                       │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓  │
│ ┃ column                                           ┃ true         ┃ true rate              ┃ hist            ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩  │
│ │ private_parking_available                        │         2661 │                   0.55 │     ▇    █      │  │
│ │ has_gps                                          │         3838 │                   0.79 │     ▂    █      │  │
│ │ has_air_conditioning                             │          978 │                    0.2 │     █    ▂      │  │
│ │ automatic_car                                    │          961 │                    0.2 │     █    ▂      │  │
│ │ has_getaround_connect                            │         2230 │                   0.46 │     █    ▇      │  │
│ │ has_speed_regulator                              │         1169 │                   0.24 │     █    ▃      │  │
│ │ winter_tires                                     │         4513 │                   0.93 │     ▁    █      │  │
│ └──────────────────────────────────────────────────┴──────────────┴────────────────────────┴─────────────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓  │
│ ┃            ┃    ┃      ┃            ┃           ┃            ┃        ┃ chars per ┃ words per  ┃ total     ┃  │
│ ┃ column     ┃ NA ┃ NA % ┃ shortest   ┃ longest   ┃ min        ┃ max    ┃ row       ┃ row        ┃ words     ┃  │
│ ┡━━━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━

In [9]:
#mask filtrer mileage > 0 : ne garder que les kilométrages positifs
price_df = price_df[price_df['mileage'] > 0]

In [10]:
# Suppression de la colonne Unnamed qui ne sert à rien

price_df = price_df.drop("Unnamed: 0", axis = "columns")


In [11]:
# Suppression des Outliers avec qui hors 3 Standard Deviation

mileage_std = price_df['mileage'].std()
mileage_mean =  price_df['mileage'].mean()
engine_power_std = price_df['engine_power'].std()
engine_power_mean = price_df['engine_power'].mean()
lower_mileage = mileage_mean - (3 * mileage_std)
upper_mileage = mileage_mean + (3 * mileage_std)
lower_engine_power = engine_power_mean - (3 * engine_power_std)
upper_engine_power = engine_power_mean + (3 * engine_power_std)

print(mileage_std, mileage_mean)

print(lower_mileage, upper_mileage)
print(lower_engine_power, upper_engine_power)

60169.013005336754 141004.15864490808
-39502.88037110219 321511.19766091835
12.20324915807413 245.78476984626383


In [12]:
upper_mileage

321511.19766091835

In [13]:
price_df['mileage'].dtypes

dtype('int64')

In [14]:
print(price_df.shape)
#upper_mileage = 321511.19766091835
price_df = price_df[(price_df['mileage'] < upper_mileage)]
print(price_df.shape)

(4841, 14)
(4795, 14)


In [15]:
print(price_df.shape)
price_df = price_df[(price_df['engine_power'] >= lower_engine_power) & (price_df['engine_power'] <= upper_engine_power)]
print(price_df.shape)

(4795, 14)
(4749, 14)


In [16]:
# Compter le nombre de voitures par model_key
count_by_model = price_df[['model_key']].value_counts()

# Compter le nombre total de modèles différents
total_models = price_df['model_key'].nunique()

print("Nombre de voitures par modèle :")
print(count_by_model)
print("\nNombre total de modèles différents :", total_models)

Nombre de voitures par modèle :
model_key  
Citroën        950
Renault        905
BMW            814
Peugeot        630
Audi           519
Nissan         274
Mitsubishi     222
Mercedes        97
Volkswagen      61
Toyota          49
SEAT            46
Subaru          37
Opel            33
PGO             33
Ferrari         33
Maserati        18
Porsche          6
Ford             5
Suzuki           4
Alfa Romeo       3
KIA Motors       3
Lamborghini      2
Fiat             2
Mazda            1
Honda            1
Yamaha           1
Name: count, dtype: int64

Nombre total de modèles différents : 26


In [17]:
# Compter le nombre de voitures par model_key
count_by_color = price_df[['paint_color']].value_counts()

# Compter le nombre total de modèles différents
total_models_color = price_df['paint_color'].nunique()

print("Nombre de voitures par modèle :")
print(count_by_color)
print("\nNombre total de couleurs différentes :", total_models_color)

Nombre de voitures par modèle :
paint_color
black          1599
grey           1153
blue            692
white           530
brown           341
silver          319
red              51
beige            41
green            17
orange            6
Name: count, dtype: int64

Nombre total de couleurs différentes : 10


In [18]:
from summarytools import dfSummary
dfSummary(price_df)


No,Variable,Stats / Values,Freqs / (% of Valid),Graph,Missing
1,model_key[object],1. Citroën2. Renault3. BMW4. Peugeot5. Audi6. Nissan7. Mitsubishi8. Mercedes9. Volkswagen10. Toyota11. other,950 (20.0%)905 (19.1%)814 (17.1%)630 (13.3%)519 (10.9%)274 (5.8%)222 (4.7%)97 (2.0%)61 (1.3%)49 (1.0%)228 (4.8%),"<img src = ""data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAJsAAAD+CAYAAAAtWHdlAAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjEwLjAsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvlHJYcgAAAAlwSFlzAAAPYQAAD2EBqD+naQAABIhJREFUeJzt3U1uGlkARtFnC6XjCQhF8jKyABbRi+1FsIDsw7Jcg+DOxD3Ij9KOHZBc3JLCOTOHQTG4eohSfeHq6elpQOF66TfA5RAbGbGRERsZsZERGxmxkREbGbGRuRpj3Iwx3i10/S9PT0+Hha5NbPXhw4e/1+v1domLT9N0f3V19Y/gLsNqvV5vd7vdYbPZPJYXfnh4eL/f77d3d3fvxhhiuwCrMcbYbDaPt7e3nxe4/s0C12QhviCQERsZsZERGxmxkREbGbGRERsZsZERGxmxkREbGbGRERuZ1Rhfny2rL7zENVnWapqm+/1+vx0LPFs2TdP9GONLfV2WYYNA5sp/mUVlyZPNqXZhFltXWVZdnkXWVZZVl2nJdZVl1YVxU5eM2MiIjYzYyIiNjNjIiI2M2MiIjYzYyIiNjNjIiI3MIoMXY5fLtNjgxdjl8ngsnIzBC5n6ZHOaXbB08GLkctmywYuRC/XgxcjlgrmpS0ZsZMRGRmxkxEZGbGTERkZsZMRGRmxkxEZGbGTERiYbvBi5kA5ejFwum8fCyRi8kPEFgcy5PkZ9XPKLs6yrrKh4yezrKisqXnOudZUVFb/wBYGM2MiIjYzYyIiNjNjIiI2M2MiIjYzYyIiNjNjIiI3M7OsqKypec5Z1lRUVL/FYOBnrKjJznWxOMo6aZfBi4MIp3jx4MXDhVHMNXgxcOMpNXTJiIyM2MmIjIzYyYiMjNjJiIyM2MmIjIzYyYiMjNjJvHrwYuHCqWQYvBi6cwmPhZAxeyJx6sjm5eLOTBi8GLczh6ODFoIW5nDp4MWjhzdzUJSM2MmIjIzYyYiMjNjJiIyM2MmIjIzYyYiMjNjJiIyM2MkfXVdZTzOWkdZX1FHOwQSBjXUXm2MnmRGM2v11XWVUxp1fXVVZVzO3Yusqqitm4qUtGbGTERkZsZMRGRmxkxEZGbGTERkZsZMRGRmxkxEbm1cGLoQtz++3gxdCFOXksnIzBC5nnJ5uTjLP53+DFwIVzuv4+eNntdodv0b315yDhRT8GL9/+NnDhbNzUJSM2MmIjIzYyYiMjNjJiIyM2MmIjIzYyYiMjNjJiI3P0F15gLj8PXgxcOCuPhZMxeCHjCwKZnz9GfYRyVj/WVZZVnNv1er3efvz4cVhWcW7XY4xxc3Pz79JvhD+fLwhkxEZGbGTERkZsZMRGRmxkxEZGbGTERkZsZMRGRmxkrscY43A4/LX0G+HPdz1N0/2nT58sqzg7j4WTsa4i42QjY/BCxuCFjMELGTd1yYiNjNjIiI2M2MiIjYzYyIiNjNjIiI2M2MiIjYzYyBi8kDF4IeOxcDIGL2Se/1Dac047ZvNj8PLSi0YwzGm1Xq+3u93usNlsHn9+4eHh4f1+v9/e3d29G2OIjTdbjTHGZrN5vL29/fzC6zfx++EP5qYuGbGRERsZsZERGxmxkREbGbGRERsZsZERGxmxkREbmdUYXx8nev7CS/8Gb7Gapul+v99vxwuPExnBMCePhZMxeCHjCwIZsZERGxmxkREbGbGRERsZsZERG5n/ANzEJE31aRAhAAAAAElFTkSuQmCC"">",0(0.0%)
2,mileage[int64],Mean (sd) : 139058.5 (54711.6)min < med < max:476.0 < 140631.0 < 321498.0IQR (CV) : 71806.0 (2.5),"4,693 distinct values","<img src = ""data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAKoAAABGCAYAAABc8A97AAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjEwLjAsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvlHJYcgAAAAlwSFlzAAAPYQAAD2EBqD+naQAAArNJREFUeJzt3E9u2kAYhvF3SkAGVCOUiux8gUpdsMymN+hhu+2ui+QCuQEbFAk5INxYOEC3rUQIeOxkvuH57Ud48Sjxn5nP7fd7AaH79NEXAJzi6qMvIFTOub6kXs3lm/1+/9zk9Vw6Qj3AOde/vr7+kabpuM761WqVO+d+EmtzCPWwXpqm49vb2+fRaFSes3C5XCZ3d3fjxWLRk0SoDSHUI0ajUTmZTP7UWNpv/GIuHA9TMIFQYQKhwgTuUVuw3W67klLnXJ3lvNo6gFAbVhRFt6qqb1mWdTqdzllvDCRebb2GUBu22WyukiQZTKfT8ubmJj9nLa+2XhdtqJ5fltLdbtf1+f3hcMirrQZFGarvl6WqqpLtdvu1LMvfkurEhoZFGao8vixJ0mw2Gz88PAxeXl46bVwczhdrqJLqf1nK85x/v4HhPSpMIFSYQKgwgVBhAqHCBEKFCYQKEwgVJhAqTCBUmECoMIFQYQKhwgRChQmEChMIFSYQKkwgVJhAqDAh6DNTHkeevY87IyzBhupz5JnjzvEJNlR5HHnmuHN8Qg5VUr0jzxx3jg8PUzCBUGECocIEQoUJhAoTgn/qvzSeY9WlSEerE2pAfMeqS/GOVifUgPiMVZfiHq1OqAHyGKsuRTpavdVQP3qOPuLRWqjM0UeT2vyLyhx9NKb1e1Tm6KMJvPCHCYQKEwgVJhAqTCBUmECoMIFQYQKhwgRChQnsnoqM58brYDddvxkqY3Xs8N14HfKm66OhMlbHFp+N16Fvun7rLypjdQzy2Hgd7Eagk+5RGatzGUI+WMjDFCQ1c7Dw6elp7Zz7JanO+qOREyok+R8snM/nn+/v779nWfaljQc5QsV/6t7f5nneb/NB7qRQl8tlcs4PS9J6vU4kqSiK5PHxcfCe663+ttXr/nd9W/4CTGGiIzLyjAYAAAAASUVORK5CYII="">",0(0.0%)
3,engine_power[int64],Mean (sd) : 127.4 (35.6)min < med < max:25.0 < 120.0 < 240.0IQR (CV) : 35.0 (3.6),51 distinct values,"<img src = ""data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAKoAAABGCAYAAABc8A97AAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjEwLjAsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvlHJYcgAAAAlwSFlzAAAPYQAAD2EBqD+naQAAAoRJREFUeJzt3TGO2kAUgOE3MbAGFFsICTouEGkLSg6Rw+YIKShScwMahEDICGctjO3tUkTsZu2xw7zZ/+sHWeIX0mPssamqSgDXfXn0BQAf0Xv0BXTFGDMUkYHFR1yrqnpp63pgx8tQjTHD6XT6PYqiSdPPOJ/PJ2PMD2J1g5ehisggiqLJarV6ieM4q7s4SZJwvV5PjsfjQEQI1QG+hioiInEcZ7PZ7HfD5cNWLwZWGKaggte/qI9iOcgxxN1BqC2zHeQY4

In [19]:
# --- Cleaning ---
# Note : ce Cleaning n'a pas donné de bons résultats. 

# 1. Traitement de la colonne 'model_key'
# ----------------------------------------
# Compter les occurrences de chaque modèle
model_counts = price_df['model_key'].value_counts()
# Identifier les modèles qui apparaissent moins de 10 fois
models_to_replace = model_counts[model_counts < 33].index
# Remplacer ces modèles par 'others_cars'
price_df.loc[price_df['model_key'].isin(models_to_replace), 'model_key'] = 'others_cars'

# --- FIN DE LA MANIPULATION DE DONNÉES ---


print("="*50)
print("ÉTAT FINAL DU DATAFRAME")
print("="*50)

print("\nDistribution de 'model_key' APRÈS regroupement:")
print(price_df['model_key'].value_counts())
print("\nDistribution de 'paint_color' APRÈS regroupement:")


ÉTAT FINAL DU DATAFRAME

Distribution de 'model_key' APRÈS regroupement:
model_key
Citroën        950
Renault        905
BMW            814
Peugeot        630
Audi           519
Nissan         274
Mitsubishi     222
Mercedes        97
Volkswagen      61
Toyota          49
others_cars     46
SEAT            46
Subaru          37
PGO             33
Opel            33
Ferrari         33
Name: count, dtype: int64

Distribution de 'paint_color' APRÈS regroupement:


## 2 Outliers

In [20]:
#Suppression de 2 Outliers Porsche
indexes_to_drop = [1796, 1925]
price_df = price_df.drop(indexes_to_drop, axis=0)
try:
    price_df.loc[indexes_to_drop]
except KeyError:
    print("Vérification réussie : Les lignes avec les index 1796 et 1925 ont bien été supprimées.")

print("\n--- Nettoyage terminé ---")

Vérification réussie : Les lignes avec les index 1796 et 1925 ont bien été supprimées.

--- Nettoyage terminé ---


# Modèles et MLFLOW

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings

In [22]:
price_df[['private_parking_available', 'has_gps', 'has_air_conditioning', 'automatic_car', 'has_getaround_connect', 'has_speed_regulator', 'winter_tires']] = price_df[['private_parking_available', 'has_gps', 'has_air_conditioning', 'automatic_car', 'has_getaround_connect', 'has_speed_regulator', 'winter_tires']].astype(int)

In [23]:
price_df

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,1,1,0,0,1,1,1,106
2,Citroën,183297,120,diesel,white,convertible,0,0,0,0,1,0,1,101
3,Citroën,128035,135,diesel,red,convertible,1,1,0,0,1,1,1,158
4,Citroën,97097,160,diesel,silver,convertible,1,1,0,0,0,1,1,183
5,Citroën,152352,225,petrol,black,convertible,1,1,0,0,1,1,1,131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4838,Toyota,39743,110,diesel,black,van,0,1,0,0,0,0,1,121
4839,Toyota,49832,100,diesel,grey,van,0,1,0,0,0,0,1,132
4840,Toyota,19633,110,diesel,grey,van,0,1,0,0,0,0,1,130
4841,Toyota,27920,110,diesel,brown,van,1,1,0,0,0,0,1,151


In [24]:
#Train Test Split

target_name = "rental_price_per_day"
Y = price_df.loc[:, target_name]
X = price_df.drop(target_name, axis=1)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


In [25]:
#Preprocessing

# Définition des colonnes numériques et catégorielles
numeric_features = ["mileage", "engine_power"]
categorical_features = ["model_key", 'fuel', "paint_color", "car_type"]

# Création du transformateur pour les données numériques
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# Création du transformateur pour les données catégorielles
categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

# Assemblage des transformateurs dans un ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="passthrough" # Laisse les autres colonnes intactes (s'il y en a)
)
print("Pré-traitement défini.\n")

Pré-traitement défini.



In [26]:
# Définir l'URI du serveur MLflow (le Hugging Face Space)
mlflow_uri = "https://ericjedha-getaroundml.hf.space"
os.environ["APP_URI"] = mlflow_uri
mlflow.set_tracking_uri(mlflow_uri)

In [27]:
import os
import pandas as pd
import numpy as np
import mlflow
from datetime import datetime

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from mlflow.models.signature import infer_signature
from dotenv import load_dotenv

# Charger les variables d'environnement depuis .env
load_dotenv()

# Charger et injecter les variables d'environnement requises
required_env_vars = [
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "AWS_DEFAULT_REGION",
    "ARTIFACT_STORE_URI"
]

for var in required_env_vars:
    val = os.getenv(var)
    if val:
        os.environ[var] = val
    else:
        print(f"[WARNING] {var} is not set in the .env file")


try:
    # Conversion uniquement pour les arrays pandas si nécessaire
    Y_train_arr = Y_train.values if hasattr(Y_train, "values") else Y_train
    Y_test_arr = Y_test.values if hasattr(Y_test, "values") else Y_test
except NameError as e:
    print(f"[ERROR] Variables de données manquantes: {e}")
    raise
# === Configuration de l'expérience ===
EXPERIMENT_NAME = "08_GETAROUND"
# Définir l'URI de suivi MLflow
mlflow.set_tracking_uri(os.environ["APP_URI"])

# Définir l'expérience
mlflow.set_experiment(EXPERIMENT_NAME)

# Récupérer les informations de l'expérience
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

## MODÈLE LINEAR REGRESSION & PIPELINE

In [ ]:
# ==============================================================================
# MODÈLE LINEAR REGRESSION & PIPELINE
# ==============================================================================
from sklearn.linear_model import LinearRegression

print("\n" + "#"*30)
print("## DÉMARRAGE DU TEST: Linear Regression ##")
print("#"*30)
print("\nDéfinition du modèle et de la pipeline...")

# Instancier le nouveau modèle
model_lr = LinearRegression()

# Créer la pipeline complète en RÉUTILISANT le même preprocessor
full_pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', model_lr)
])

# NOTE : La régression linéaire simple n'a pas d'hyperparamètres majeurs à optimiser.
# On garde donc une grille vide. GridSearchCV servira ici principalement à effectuer
# la validation croisée de manière propre et à garder un workflow cohérent.
param_grid_lr = {}

# Configurer GridSearchCV pour le modèle linéaire
grid_search_lr = GridSearchCV(
    estimator=full_pipeline_lr,
    param_grid=param_grid_lr,
    cv=3,
    n_jobs=-1,
    scoring='r2'
)
print("Modèle et pipeline prêts pour l'entraînement.\n")


# ==============================================================================
# MLFLOW POUR LINEAR REGRESSION
# ==============================================================================
print("Lancement de l'expérimentation MLflow pour Linear Regression...")

mlflow.sklearn.autolog(log_model_signatures=True, log_models=True)

run_name_lr = "Run_LinearRegression_" + datetime.now().strftime("%Y%m%d_%H%M%S")

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name_lr) as run:
    # --- Entraînement ---
    grid_search_lr.fit(X_train, Y_train)

    # --- Prédictions ---
    y_train_pred_lr = grid_search_lr.predict(X_train)
    y_test_pred_lr = grid_search_lr.predict(X_test)

    # --- Calcul des métriques ---
    r2_train_lr = r2_score(Y_train, y_train_pred_lr)
    mae_train_lr = mean_absolute_error(Y_train, y_train_pred_lr)
    rmse_train_lr = np.sqrt(mean_squared_error(Y_train, y_train_pred_lr))

    r2_test_lr = r2_score(Y_test, y_test_pred_lr)
    mae_test_lr = mean_absolute_error(Y_test, y_test_pred_lr)
    rmse_test_lr = np.sqrt(mean_squared_error(Y_test, y_test_pred_lr))

    cv_score_lr = grid_search_lr.best_score_
    
    # --- Logging MLflow ---
    print("\nEnregistrement des résultats dans MLflow...")
    mlflow.log_params(grid_search_lr.best_params_) # Sera vide, ce qui est normal

    mlflow.log_metric("R2_train", r2_train_lr)
    mlflow.log_metric("MAE_train", mae_train_lr)
    mlflow.log_metric("RMSE_train", rmse_train_lr)

    mlflow.log_metric("R2_test", r2_test_lr)
    mlflow.log_metric("MAE_test", mae_test_lr)
    mlflow.log_metric("RMSE_test", rmse_test_lr)
    
    mlflow.log_metric("CV_score_R2", cv_score_lr)

    # --- Sauvegarde du modèle (la pipeline complète) ---
    signature_lr = infer_signature(X_train, y_train_pred_lr)
    mlflow.sklearn.log_model(
        sk_model=grid_search_lr.best_estimator_,
        artifact_path="linear_regression_pipeline",
        signature=signature_lr
    )
    print("Modèle et métriques sauvegardés.")

    # ==============================================================================
    # AFFICHAGE DES RÉSULTATS POUR LINEAR REGRESSION
    # ==============================================================================
    print("\n" + "="*60)
    print("📈 LINEAR REGRESSION - RÉSULTATS FINAUX 📈")
    print(f"🎯 Run ID: {run.info.run_id}")
    print("="*60)
    
    print("\n🔧 HYPERPARAMÈTRES:")
    print("  • Aucun hyperparamètre à optimiser pour ce modèle.")
    
    print(f"\n📊 SCORE R² MOYEN (Validation Croisée): {cv_score_lr:.4f}")
    
    print("\n📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:")
    print(f"  • R² Score: {r2_train_lr:.4f}")
    print(f"  • MAE:      {mae_train_lr:.2f} ")
    print(f"  • RMSE:     {rmse_train_lr:.2f} ")
    
    print("\n🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:")
    print(f"  • R² Score: {r2_test_lr:.4f}")
    print(f"  • MAE:      {mae_test_lr:.2f} ")
    print(f"  • RMSE:     {rmse_test_lr:.2f} ")
    
    overfitting_r2_diff_lr = r2_train_lr - r2_test_lr
    if abs(overfitting_r2_diff_lr) > 0.1: # On prend la valeur absolue car un modèle linéaire peut sous-performer sur le train
        print(f"\n⚠️  ATTENTION: Écart de performance notable (écart R²: {overfitting_r2_diff_lr:.4f})")
    else:
        print(f"\n✅ Modèle stable (écart R²: {overfitting_r2_diff_lr:.4f})")
    
    print("="*60)
    print(f"✅ Expérimentation terminée. Comparez les résultats avec les autres modèles dans MLflow UI.")
    print("="*60)


##############################
## DÉMARRAGE DU TEST: Linear Regression ##
##############################

Définition du modèle et de la pipeline...
Modèle et pipeline prêts pour l'entraînement.

Lancement de l'expérimentation MLflow pour Linear Regression...


2025/08/15 21:38:28 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Foun

🏃 View run tasteful-boar-930 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/6ffec1c3e8434a17a2d29a107d093a2a
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20


2025/08/15 21:38:49 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/08/15 21:38:50 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/


Enregistrement des résultats dans MLflow...


/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Modèle et métriques sauvegardés.

📈 LINEAR REGRESSION - RÉSULTATS FINAUX 📈
🎯 Run ID: 55e04ecd3744413db1b5d5b35294a7d5

🔧 HYPERPARAMÈTRES:
  • Aucun hyperparamètre à optimiser pour ce modèle.

📊 SCORE R² MOYEN (Validation Croisée): 0.6747

📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:
  • R² Score: 0.6901
  • MAE:      11.96 
  • RMSE:     17.84 

🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:
  • R² Score: 0.7267
  • MAE:      12.15 
  • RMSE:     16.95 

✅ Modèle stable (écart R²: -0.0366)
✅ Expérimentation terminée. Comparez les résultats avec les autres modèles dans MLflow UI.
🏃 View run Run_LinearRegression_20250815_213825 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/55e04ecd3744413db1b5d5b35294a7d5
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20



# XGBOOST & SA PIPELINE


In [28]:
# ==============================================================================
# XGBOOST & SA PIPELINE
# ==============================================================================
import xgboost as xgb

print("\n" + "#"*30)
print("## DÉMARRAGE DU TEST: XGBoost ##")
print("#"*30)
print("\nDéfinition du modèle et de la grille de recherche...")

model_xgb = xgb.XGBRegressor(random_state=42)

# Pipeline avec le même preprocessor
full_pipeline_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', model_xgb)
])

# hyperparamètres pour XGBoost

param_grid_xgb = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [3, 5, 7],
    'regressor__learning_rate': [0.05, 0.1]
}

# GridSearchCV pour le modèle XGBoost
grid_search_xgb = GridSearchCV(
    estimator=full_pipeline_xgb,
    param_grid=param_grid_xgb,
    cv=3,
    n_jobs=-1,
    verbose=2,
    scoring='r2'
)
print("Modèle et pipeline prêts pour l'entraînement.\n")


# ==============================================================================
# 3. (TER) EXPÉRIMENTATION MLFLOW POUR XGBOOST
# ==============================================================================
print("Lancement de l'expérimentation MLflow pour XGBoost...")

run_name_xgb = "Run_XGBoost_GridS_CV5" + datetime.now().strftime("%Y%m%d_%H%M%S")

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name_xgb) as run:
    # --- Entraînement ---
    grid_search_xgb.fit(X_train, Y_train)

    # --- Prédictions ---
    y_train_pred_xgb = grid_search_xgb.predict(X_train)
    y_test_pred_xgb = grid_search_xgb.predict(X_test)

    # --- Calcul des métriques ---
    r2_train_xgb = r2_score(Y_train, y_train_pred_xgb)
    mae_train_xgb = mean_absolute_error(Y_train, y_train_pred_xgb)
    rmse_train_xgb = np.sqrt(mean_squared_error(Y_train, y_train_pred_xgb))

    r2_test_xgb = r2_score(Y_test, y_test_pred_xgb)
    mae_test_xgb = mean_absolute_error(Y_test, y_test_pred_xgb)
    rmse_test_xgb = np.sqrt(mean_squared_error(Y_test, y_test_pred_xgb))
    
    cv_score_xgb = grid_search_xgb.best_score_
    
    # --- Logging MLflow ---
    print("\nEnregistrement des résultats dans MLflow...")
    mlflow.log_params(grid_search_xgb.best_params_)

    mlflow.log_metric("R2_train", r2_train_xgb)
    mlflow.log_metric("MAE_train", mae_train_xgb)
    mlflow.log_metric("RMSE_train", rmse_train_xgb)

    mlflow.log_metric("R2_test", r2_test_xgb)
    mlflow.log_metric("MAE_test", mae_test_xgb)
    mlflow.log_metric("RMSE_test", rmse_test_xgb)
    
    mlflow.log_metric("CV_score_R2", cv_score_xgb)

    # --- Sauvegarde du modèle (la pipeline complète) ---
    signature_xgb = infer_signature(X_train, y_train_pred_xgb)
    mlflow.sklearn.log_model(
        sk_model=grid_search_xgb.best_estimator_,
        artifact_path="xgboost_pipeline",
        signature=signature_xgb
    )
    print("Modèle et métriques sauvegardés.")

    # ==============================================================================
    # 4. (TER) AFFICHAGE DES RÉSULTATS POUR XGBOOST
    # ==============================================================================
    print("\n" + "="*60)
    print("🚀 XGBOOST - RÉSULTATS FINAUX 🚀")
    print(f"🎯 Run ID: {run.info.run_id}")
    print("="*60)
    
    print("\n🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:")
    for param, value in grid_search_xgb.best_params_.items():
        clean_param = param.replace('regressor__', '')
        print(f"  • {clean_param}: {value}")
    
    print(f"\n📊 MEILLEUR SCORE R² (Validation Croisée): {cv_score_xgb:.4f}")
    
    print("\n📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:")
    print(f"  • R² Score: {r2_train_xgb:.4f}")
    print(f"  • MAE:      {mae_train_xgb:.2f} $")
    print(f"  • RMSE:     {rmse_train_xgb:.2f} $")
    
    print("\n🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:")
    print(f"  • R² Score: {r2_test_xgb:.4f}")
    print(f"  • MAE:      {mae_test_xgb:.2f} $")
    print(f"  • RMSE:     {rmse_test_xgb:.2f} $")
    
    overfitting_r2_diff_xgb = r2_train_xgb - r2_test_xgb
    if overfitting_r2_diff_xgb > 0.1:
        print(f"\n⚠️  ATTENTION: Possible surapprentissage détecté (écart R²: {overfitting_r2_diff_xgb:.4f})")
    else:
        print(f"\n✅ Modèle bien généralisé (écart R²: {overfitting_r2_diff_xgb:.4f})")
    
    print("="*60)
    print(f"✅ Expérimentation terminée. Comparez les résultats dans MLflow UI.")
    print("="*60)


##############################
## DÉMARRAGE DU TEST: XGBoost ##
##############################

Définition du modèle et de la grille de recherche...
Modèle et pipeline prêts pour l'entraînement.

Lancement de l'expérimentation MLflow pour XGBoost...
Fitting 3 folds for each of 18 candidates, totalling 54 fits


/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
 

[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=200; total time=   0.2s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=200; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=200; total time=   0.2s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=300; total time=   0.2s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=300; total time=   0.2s


/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
 

[CV] END regressor__learning_rate=0.05, regressor__max_depth=5, regressor__n_estimators=100; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=5, regressor__n_estimators=100; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=5, regressor__n_estimators=100; total time=   0.2s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=300; total time=   0.2s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=5, regressor__n_estimators=200; total time=   0.3s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=5, regressor__n_estimators=200; total time=   0.3s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=5, regressor__n_estimators=200; total time=   0.3s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=7, regressor__n_estimators=100; total time=   0.3s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=7, regressor__n_estimators=100; total time=

...

[CV] END regressor__learning_rate=0.1, regressor__max_depth=5, regressor__n_estimators=200; total time=   0.2s
[CV] END regressor__learning_rate=0.1, regressor__max_depth=5, regressor__n_estimators=200; total time=   0.2s
[CV] END regressor__learning_rate=0.1, regressor__max_depth=7, regressor__n_estimators=100; total time=   0.2s
[CV] END regressor__learning_rate=0.1, regressor__max_depth=7, regressor__n_estimators=100; total time=   0.2s
[CV] END regressor__learning_rate=0.1, regressor__max_depth=5, regressor__n_estimators=300; total time=   0.3s
[CV] END regressor__learning_rate=0.1, regressor__max_depth=5, regressor__n_estimators=300; total time=   0.3s
[CV] END regressor__learning_rate=0.1, regressor__max_depth=5, regressor__n_estimators=300; total time=   0.3s
[CV] END regressor__learning_rate=0.1, regressor__max_depth=7, regressor__n_estimators=100; total time=   0.2s


/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
 

[CV] END regressor__learning_rate=0.1, regressor__max_depth=7, regressor__n_estimators=200; total time=   0.4s
[CV] END regressor__learning_rate=0.1, regressor__max_depth=7, regressor__n_estimators=200; total time=   0.4s
[CV] END regressor__learning_rate=0.1, regressor__max_depth=7, regressor__n_estimators=200; total time=   0.3s
[CV] END regressor__learning_rate=0.1, regressor__max_depth=7, regressor__n_estimators=300; total time=   0.4s
[CV] END regressor__learning_rate=0.1, regressor__max_depth=7, regressor__n_estimators=300; total time=   0.4s
[CV] END regressor__learning_rate=0.1, regressor__max_depth=7, regressor__n_estimators=300; total time=   0.4s

Enregistrement des résultats dans MLflow...


/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Modèle et métriques sauvegardés.

🚀 XGBOOST - RÉSULTATS FINAUX 🚀
🎯 Run ID: 1bb55a0130904a569f0180bcad36a649

🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:
  • learning_rate: 0.05
  • max_depth: 5
  • n_estimators: 300

📊 MEILLEUR SCORE R² (Validation Croisée): 0.7266

📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:
  • R² Score: 0.8512
  • MAE:      8.20 $
  • RMSE:     12.36 $

🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:
  • R² Score: 0.7924
  • MAE:      10.12 $
  • RMSE:     14.77 $

✅ Modèle bien généralisé (écart R²: 0.0589)
✅ Expérimentation terminée. Comparez les résultats dans MLflow UI.
🏃 View run Run_XGBoost_GridS_CV520250820_110229 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/1bb55a0130904a569f0180bcad36a649
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20


# MODÈLE XGBOOST AVEC RÉGULARISATION LASSO (L1)

In [ ]:
# ==============================================================================
#  XGBOOST AVEC RÉGULARISATION LASSO (L1)
# ==============================================================================
import xgboost as xgb

print("\n" + "#"*30)
print("## DÉMARRAGE DU TEST: XGBoost avec Régularisation L1 (Lasso) ##")
print("#"*30)
print("\nDéfinition du modèle et de la grille de recherche...")

model_xgb = xgb.XGBRegressor(random_state=42)

full_pipeline_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', model_xgb)
])

# MISE À JOUR DE LA GRILLE DE RECHERCHE
# On ajoute 'regressor__reg_alpha' pour tester différentes forces de régularisation L1.
param_grid_xgb_lasso = {
    'regressor__n_estimators': [100, 300],
    'regressor__max_depth': [3, 5],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__reg_alpha': [0, 0.1, 0.5, 1]  # 0 = pas de L1, 1 = L1 assez forte
}

# Configurer GridSearchCV avec la nouvelle grille
grid_search_xgb_lasso = GridSearchCV(
    estimator=full_pipeline_xgb,
    param_grid=param_grid_xgb_lasso,
    cv=3,
    n_jobs=-1,
    verbose=2,
    scoring='r2'
)
print("Modèle et pipeline prêts pour l'entraînement.\n")


# ==============================================================================
# MLFLOW POUR XGBOOST AVEC LASSO
# ==============================================================================
print("Lancement de l'expérimentation MLflow pour XGBoost avec Lasso...")

run_name_xgb_lasso = "Run_XGBoost_Lasso_" + datetime.now().strftime("%Y%m%d_%H%M%S")

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name_xgb_lasso) as run:
    # --- Entraînement ---
    grid_search_xgb_lasso.fit(X_train, Y_train)

    # ... (le reste du code pour les prédictions, le calcul des métriques et le logging est identique) ...
    # On utilise simplement la nouvelle variable `grid_search_xgb_lasso`

    y_train_pred_xgb = grid_search_xgb_lasso.predict(X_train)
    y_test_pred_xgb = grid_search_xgb_lasso.predict(X_test)
    
    r2_train_xgb = r2_score(Y_train, y_train_pred_xgb)
    mae_train_xgb = mean_absolute_error(Y_train, y_train_pred_xgb)
    rmse_train_xgb = np.sqrt(mean_squared_error(Y_train, y_train_pred_xgb))
    
    r2_test_xgb = r2_score(Y_test, y_test_pred_xgb)
    mae_test_xgb = mean_absolute_error(Y_test, y_test_pred_xgb)
    rmse_test_xgb = np.sqrt(mean_squared_error(Y_test, y_test_pred_xgb))
    
    cv_score_xgb = grid_search_xgb_lasso.best_score_
    
    print("\nEnregistrement des résultats dans MLflow...")
    mlflow.log_params(grid_search_xgb_lasso.best_params_)
    mlflow.log_metric("R2_train", r2_train_xgb)
    mlflow.log_metric("MAE_train", mae_train_xgb)
    mlflow.log_metric("RMSE_train", rmse_train_xgb)
    mlflow.log_metric("R2_test", r2_test_xgb)
    mlflow.log_metric("MAE_test", mae_test_xgb)
    mlflow.log_metric("RMSE_test", rmse_test_xgb)
    mlflow.log_metric("CV_score_R2", cv_score_xgb)

    signature_xgb = infer_signature(X_train, y_train_pred_xgb)
    mlflow.sklearn.log_model(
        sk_model=grid_search_xgb_lasso.best_estimator_,
        artifact_path="xgboost_lasso_pipeline",
        signature=signature_xgb
    )
    print("Modèle et métriques sauvegardés.")

    # ==============================================================================
    # AFFICHAGE DES RÉSULTATS POUR XGBOOST AVEC LASSO
    # ==============================================================================
    print("\n" + "="*60)
    print("🚀 XGBOOST avec Régularisation L1 (Lasso) - RÉSULTATS 🚀")
    print(f"🎯 Run ID: {run.info.run_id}")
    print("="*60)
    
    print("\n🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:")
    for param, value in grid_search_xgb_lasso.best_params_.items():
        clean_param = param.replace('regressor__', '')
        print(f"  • {clean_param}: {value}")
    
    print(f"\n📊 MEILLEUR SCORE R² (Validation Croisée): {cv_score_xgb:.4f}")
    
    print("\n📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:")
    print(f"  • R² Score: {r2_train_xgb:.4f}")
    print(f"  • MAE:      {mae_train_xgb:.2f} $")
    print(f"  • RMSE:     {rmse_train_xgb:.2f} $")
    
    print("\n🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:")
    print(f"  • R² Score: {r2_test_xgb:.4f}")
    print(f"  • MAE:      {mae_test_xgb:.2f} $")
    print(f"  • RMSE:     {rmse_test_xgb:.2f} $")
    
    overfitting_r2_diff_xgb = r2_train_xgb - r2_test_xgb
    if overfitting_r2_diff_xgb > 0.1:
        print(f"\n⚠️  ATTENTION: Possible surapprentissage détecté (écart R²: {overfitting_r2_diff_xgb:.4f})")
    else:
        print(f"\n✅ Modèle bien généralisé (écart R²: {overfitting_r2_diff_xgb:.4f})")
    
    print("="*60)
    print(f"✅ Expérimentation terminée. Comparez les résultats dans MLflow UI.")
    print("="*60)


##############################
## DÉMARRAGE DU TEST: XGBoost avec Régularisation L1 (Lasso) ##
##############################

Définition du modèle et de la grille de recherche...
Modèle et pipeline prêts pour l'entraînement.

Lancement de l'expérimentation MLflow pour XGBoost avec Lasso...


2025/08/15 21:45:11 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Foun

Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100, regressor__reg_alpha=0; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100, regressor__reg_alpha=0; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100, regressor__reg_alpha=0; total time=   0.1s


/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
 

[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100, regressor__reg_alpha=0.1; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100, regressor__reg_alpha=0.1; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100, regressor__reg_alpha=0.1; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100, regressor__reg_alpha=0.5; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100, regressor__reg_alpha=0.5; total time=   0.1s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100, regressor__reg_alpha=0.5; total time=   0.2s
[CV] END regressor__learning_rate=0.05, regressor__max_depth=3, regressor__n_estimators=100, regressor__reg_alpha=1; total time=   0.2s
[CV] END regressor__learning_rate=0.

...

2025/08/15 21:45:14 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).
"
2025/08/15 21:45:15 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot rep

🏃 View run legendary-tern-897 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/5da7e0a351a24e4a9e11aa4242a0cd8b
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run grandiose-bat-280 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/0be7484fc82f4a2289495eaf81ca1424
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run respected-auk-778 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/68e33962cce54e3586939a3c4eeada24
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run dashing-duck-69 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/b46b523a5bd74ccebb4822f5971d25d5
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run legendary-whale-560 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/4f68af9f110944e99f86d39ef31b40e1
🧪 View experiment at: https://ericjedha-getaroundml.


Enregistrement des résultats dans MLflow...


/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Modèle et métriques sauvegardés.

🚀 XGBOOST avec Régularisation L1 (Lasso) - RÉSULTATS 🚀
🎯 Run ID: b4c5453d597d48fca32d4be6097d450f

🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:
  • learning_rate: 0.1
  • max_depth: 5
  • n_estimators: 300
  • reg_alpha: 1

📊 MEILLEUR SCORE R² (Validation Croisée): 0.7300

📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:
  • R² Score: 0.8932
  • MAE:      7.06 $
  • RMSE:     10.48 $

🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:
  • R² Score: 0.7870
  • MAE:      10.06 $
  • RMSE:     14.96 $

⚠️  ATTENTION: Possible surapprentissage détecté (écart R²: 0.1061)
✅ Expérimentation terminée. Comparez les résultats dans MLflow UI.
🏃 View run Run_XGBoost_Lasso_20250815_214509 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/b4c5453d597d48fca32d4be6097d450f
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20


# Random Forest

In [29]:
#Random Forest

model = RandomForestRegressor(random_state=42)

# Créer la pipeline complète qui enchaîne le pré-traitement et le modèle
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', model)
])

# Définir la grille d'hyperparamètres à tester pour le modèle
# IMPORTANT : les noms des paramètres doivent être préfixés par le nom de l'étape dans la pipeline ('regressor__')
param_grid = {
    "regressor__max_depth": [2, 10, 20, 30],
    "regressor__min_samples_leaf": [1, 2, 5],
    "regressor__min_samples_split":  [2, 4, 8],
    "regressor__n_estimators": [100, 150, 200]
}

# Configurer la recherche par grille (GridSearchCV) sur la pipeline complète
grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    verbose=2,
    scoring='r2' # On peut spécifier une métrique pour l'optimisation
)
print("Modèle et pipeline prêts pour l'entraînement.\n")


# ==============================================================================
# MLFLOW (TRAINING ET LOGGING)
# ==============================================================================


run_name = "Run_RandomForest_GridSearch_" + datetime.now().strftime("%Y%m%d_%H%M%S")

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name) as run:
    # --- Entraînement ---
    # On entraîne GridSearchCV sur les données BRUTES. La pipeline gère le pré-traitement.
    grid_search.fit(X_train, Y_train)

    # --- Prédictions ---
    y_train_pred = grid_search.predict(X_train)
    y_test_pred = grid_search.predict(X_test)

    # --- Calcul des métriques ---
    r2_train = r2_score(Y_train, y_train_pred)
    mae_train = mean_absolute_error(Y_train, y_train_pred)
    rmse_train = np.sqrt(mean_squared_error(Y_train, y_train_pred))

    r2_test = r2_score(Y_test, y_test_pred)
    mae_test = mean_absolute_error(Y_test, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(Y_test, y_test_pred))

    # --- Logging MLflow ---
    print("\nEnregistrement des résultats dans MLflow...")
    mlflow.log_params(grid_search.best_params_)

    mlflow.log_metric("R2_train", r2_train)
    mlflow.log_metric("MAE_train", mae_train)
    mlflow.log_metric("RMSE_train", rmse_train)

    mlflow.log_metric("R2_test", r2_test)
    mlflow.log_metric("MAE_test", mae_test)
    mlflow.log_metric("RMSE_test", rmse_test)
    
    mlflow.log_metric("CV_best_score_R2", grid_search.best_score_)


    # --- Sauvegarde du modèle (la pipeline complète) ---
    # La signature est inférée sur les données d'entrée brutes (X_train)
    signature = infer_signature(X_train, y_train_pred)
    mlflow.sklearn.log_model(
        sk_model=grid_search.best_estimator_,
        artifact_path="model_pipeline", # L'artefact est la pipeline entière
        signature=signature
    )
    print("Modèle et métriques sauvegardés.")

    # ==============================================================================
    # AFFICHAGE DES RÉSULTATS DANS LA CONSOLE
    # ==============================================================================
    print("\n" + "="*60)
    print("🌲 RANDOM FOREST REGRESSOR - RÉSULTATS FINAUX 🌲")
    print(f"🎯 Run ID: {run.info.run_id}")
    print("="*60)
    
    print("\n🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:")
    for param, value in grid_search.best_params_.items():
        # On retire le préfixe 'regressor__' pour un affichage plus propre
        clean_param = param.replace('regressor__', '')
        print(f"  • {clean_param}: {value}")
    
    print(f"\n📊 MEILLEUR SCORE R² (Validation Croisée): {grid_search.best_score_:.4f}")
    
    print("\n📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:")
    print(f"  • R² Score: {r2_train:.4f}")
    print(f"  • MAE:      {mae_train:.2f}")
    print(f"  • RMSE:     {rmse_train:.2f}")
    
    print("\n🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:")
    print(f"  • R² Score: {r2_test:.4f}")
    print(f"  • MAE:      {mae_test:.2f}")
    print(f"  • RMSE:     {rmse_test:.2f}")
    
    # Analyse rapide du surapprentissage
    overfitting_r2_diff = r2_train - r2_test
    if overfitting_r2_diff > 0.1:
        print(f"\n⚠️  ATTENTION: Possible surapprentissage détecté (écart R²: {overfitting_r2_diff:.4f})")
    else:
        print(f"\n✅ Modèle bien généralisé (écart R²: {overfitting_r2_diff:.4f})")
    
    print("="*60)
    print(f"✅ Expérimentation terminée. Retrouvez les détails dans MLflow UI.")
    print("="*60)

Modèle et pipeline prêts pour l'entraînement.



2025/08/15 21:51:52 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fitting 3 folds for each of 108 candidates, totalling 324 fits


/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
 

[CV] END regressor__max_depth=2, regressor__min_samples_leaf=1, regressor__min_samples_split=2, regressor__n_estimators=100; total time=   0.5s
[CV] END regressor__max_depth=2, regressor__min_samples_leaf=1, regressor__min_samples_split=2, regressor__n_estimators=100; total time=   0.5s
[CV] END regressor__max_depth=2, regressor__min_samples_leaf=1, regressor__min_samples_split=2, regressor__n_estimators=100; total time=   0.5s
[CV] END regressor__max_depth=2, regressor__min_samples_leaf=1, regressor__min_samples_split=2, regressor__n_estimators=150; total time=   0.6s
[CV] END regressor__max_depth=2, regressor__min_samples_leaf=1, regressor__min_samples_split=2, regressor__n_estimators=150; total time=   0.7s
[CV] END regressor__max_depth=2, regressor__min_samples_leaf=1, regressor__min_samples_split=2, regressor__n_estimators=150; total time=   0.7s


...

[CV] END regressor__max_depth=30, regressor__min_samples_leaf=5, regressor__min_samples_split=8, regressor__n_estimators=200; total time=   5.4s


2025/08/15 21:55:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).
"
2025/08/15 21:55:35 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot rep


Enregistrement des résultats dans MLflow...


/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Modèle et métriques sauvegardés.

🌲 RANDOM FOREST REGRESSOR - RÉSULTATS FINAUX 🌲
🎯 Run ID: 5198b078007c4cca81910fa5af22013f

🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:
  • max_depth: 30
  • min_samples_leaf: 1
  • min_samples_split: 4
  • n_estimators: 200

📊 MEILLEUR SCORE R² (Validation Croisée): 0.7104

📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:
  • R² Score: 0.9426
  • MAE:      4.68
  • RMSE:     7.68

🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:
  • R² Score: 0.7948
  • MAE:      10.09
  • RMSE:     14.69

⚠️  ATTENTION: Possible surapprentissage détecté (écart R²: 0.1477)
✅ Expérimentation terminée. Retrouvez les détails dans MLflow UI.
🏃 View run Run_RandomForest_GridSearch_20250815_215151 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/5198b078007c4cca81910fa5af22013f
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20


# XGBOOST - AVEC RÉGULARISATION RIDGE (L2)

In [35]:
# ==============================================================================
# XGBOOST - AVEC RÉGULARISATION RIDGE (L2)
# ==============================================================================
import xgboost as xgb

print("\n" + "#"*30)
print("## NOUVEL ESSAI: XGBoost avec Régularisation L2 (Ridge) ##")
print("#"*30)
print("\nDéfinition du modèle et de la grille de recherche...")

model_xgb = xgb.XGBRegressor(random_state=42)

full_pipeline_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', model_xgb)
])

# NOUVELLE GRILLE DE RECHERCHE - FOCUS SUR RIDGE (reg_lambda)
# On conserve les contraintes sur la profondeur des arbres qui est le levier le plus puissant.
param_grid_xgb_ridge = {
    'regressor__max_depth': [3, 4, 5, 6],
    'regressor__learning_rate': [0.001, 0.05, 0.1, 0.2],
    'regressor__n_estimators': [250, 300, 400],
    # On teste différentes forces de régularisation L2 (Ridge)
    'regressor__reg_lambda': [1, 5, 10]  # On teste des valeurs de plus en plus fortes
}

# Configurer GridSearchCV avec la nouvelle grille
grid_search_xgb_ridge = GridSearchCV(
    estimator=full_pipeline_xgb,
    param_grid=param_grid_xgb_ridge,
    cv=5,
    n_jobs=-1,
    verbose=2,
    scoring='r2'
)
print("Modèle et pipeline prêts pour l'entraînement.\n")


# ==============================================================================
# EXPÉRIMENTATION MLFLOW POUR XGBOOST AVEC RIDGE
# ==============================================================================
print("Lancement de l'expérimentation MLflow pour XGBoost avec Ridge...")

run_name_xgb_ridge = "Run_XGBoost_Ridge_CV5" + datetime.now().strftime("%Y%m%d_%H%M%S")

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name_xgb_ridge) as run:
    # --- Entraînement ---
    grid_search_xgb_ridge.fit(X_train, Y_train)

    # --- Prédictions, Métriques, Logging... ---
    # (Le code suivant est identique au précédent, on a juste changé le nom de la variable grid_search)
    y_train_pred_xgb = grid_search_xgb_ridge.predict(X_train)
    y_test_pred_xgb = grid_search_xgb_ridge.predict(X_test)
    
    r2_train_xgb = r2_score(Y_train, y_train_pred_xgb)
    mae_train_xgb = mean_absolute_error(Y_train, y_train_pred_xgb)
    rmse_train_xgb = np.sqrt(mean_squared_error(Y_train, y_train_pred_xgb))
    
    r2_test_xgb = r2_score(Y_test, y_test_pred_xgb)
    mae_test_xgb = mean_absolute_error(Y_test, y_test_pred_xgb)
    rmse_test_xgb = np.sqrt(mean_squared_error(Y_test, y_test_pred_xgb))
    
    cv_score_xgb = grid_search_xgb_ridge.best_score_
    
    print("\nEnregistrement des résultats dans MLflow...")
    mlflow.log_params(grid_search_xgb_ridge.best_params_)
    mlflow.log_metric("R2_train", r2_train_xgb)
    mlflow.log_metric("MAE_train", mae_train_xgb)
    mlflow.log_metric("RMSE_train", rmse_train_xgb)
    mlflow.log_metric("R2_test", r2_test_xgb)
    mlflow.log_metric("MAE_test", mae_test_xgb)
    mlflow.log_metric("RMSE_test", rmse_test_xgb)
    mlflow.log_metric("CV_score_R2", cv_score_xgb)

    signature_xgb = infer_signature(X_train, y_train_pred_xgb)
    mlflow.sklearn.log_model(
        sk_model=grid_search_xgb_ridge.best_estimator_,
        artifact_path="xgboost_ridge_pipeline",
        signature=signature_xgb
    )
    print("Modèle et métriques sauvegardés.")

    # ==============================================================================
    # AFFICHAGE DES RÉSULTATS POUR XGBOOST AVEC RIDGE
    # ==============================================================================
    print("\n" + "="*60)
    print("🚀 XGBOOST avec Régularisation L2 (Ridge) - RÉSULTATS 🚀")
    print(f"🎯 Run ID: {run.info.run_id}")
    print("="*60)
    
    print("\n🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:")
    for param, value in grid_search_xgb_ridge.best_params_.items():
        clean_param = param.replace('regressor__', '')
        print(f"  • {clean_param}: {value}")
    
    print(f"\n📊 MEILLEUR SCORE R² (Validation Croisée): {cv_score_xgb:.4f}")
    
    print("\n📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:")
    print(f"  • R² Score: {r2_train_xgb:.4f}")
    print(f"  • MAE:      {mae_train_xgb:.2f} $")
    print(f"  • RMSE:     {rmse_train_xgb:.2f} $")
    
    print("\n🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:")
    print(f"  • R² Score: {r2_test_xgb:.4f}")
    print(f"  • MAE:      {mae_test_xgb:.2f} $")
    print(f"  • RMSE:     {rmse_test_xgb:.2f} $")
    
    overfitting_r2_diff_xgb = r2_train_xgb - r2_test_xgb
    if overfitting_r2_diff_xgb > 0.1:
        print(f"\n⚠️  ATTENTION: Surapprentissage encore présent (écart R²: {overfitting_r2_diff_xgb:.4f})")
    else:
        print(f"\n✅ Modèle bien généralisé (écart R²: {overfitting_r2_diff_xgb:.4f})")
    
    print("="*60)
    print(f"✅ Expérimentation terminée. Comparez les résultats dans MLflow UI.")
    print("="*60)


##############################
## NOUVEL ESSAI: XGBoost avec Régularisation L2 (Ridge) ##
##############################

Définition du modèle et de la grille de recherche...
Modèle et pipeline prêts pour l'entraînement.

Lancement de l'expérimentation MLflow pour XGBoost avec Ridge...


2025/08/15 22:38:03 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fitting 5 folds for each of 144 candidates, totalling 720 fits
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=1; total time=   0.3s
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=1; total time=   0.3s
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=1; total time=   0.3s
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=1; total time=   0.3s
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=5; total time=   0.3s
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=1; total time=   0.3s
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_

/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
 

[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=5; total time=   0.3s
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=5; total time=   0.3s
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=10; total time=   0.3s
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=10; total time=   0.3s
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=10; total time=   0.3s
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=10; total time=   0.3s
[CV] END regressor__learning_rate=0.001, regressor__max_depth=3, regressor__n_estimators=250, regressor__reg_lambda=10; total time=   0.3s
[CV] END regressor__learning_

...

[CV] END regressor__learning_rate=0.2, regressor__max_depth=6, regressor__n_estimators=400, regressor__reg_lambda=10; total time=   0.8s
[CV] END regressor__learning_rate=0.2, regressor__max_depth=6, regressor__n_estimators=400, regressor__reg_lambda=10; total time=   0.8s
[CV] END regressor__learning_rate=0.2, regressor__max_depth=6, regressor__n_estimators=400, regressor__reg_lambda=10; total time=   0.7s
[CV] END regressor__learning_rate=0.2, regressor__max_depth=6, regressor__n_estimators=400, regressor__reg_lambda=10; total time=   0.9s


2025/08/15 22:39:16 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).
"
2025/08/15 22:39:17 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot rep


Enregistrement des résultats dans MLflow...


/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Modèle et métriques sauvegardés.

🚀 XGBOOST avec Régularisation L2 (Ridge) - RÉSULTATS 🚀
🎯 Run ID: 7a25968cdbfa478db77308989de224fa

🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:
  • learning_rate: 0.1
  • max_depth: 5
  • n_estimators: 300
  • reg_lambda: 1

📊 MEILLEUR SCORE R² (Validation Croisée): 0.7419

📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:
  • R² Score: 0.8935
  • MAE:      7.02 $
  • RMSE:     10.46 $

🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:
  • R² Score: 0.7928
  • MAE:      9.99 $
  • RMSE:     14.76 $

⚠️  ATTENTION: Surapprentissage encore présent (écart R²: 0.1007)
✅ Expérimentation terminée. Comparez les résultats dans MLflow UI.
🏃 View run Run_XGBoost_Ridge_CV520250815_223801 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/7a25968cdbfa478db77308989de224fa
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20


# LightGBM

In [30]:
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import mlflow
from datetime import datetime
import numpy as np

# ==============================================================================
# PREPROCESSING
# ==============================================================================
# Définition des colonnes numériques et catégorielles
numeric_features = ["mileage", "engine_power"]
categorical_features = ["model_key", "fuel", "paint_color", "car_type"]

# Création des transformateurs
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])

# Assemblage des transformateurs dans un ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="passthrough"
)
# Active la sortie en DataFrame pour conserver les noms de colonnes
preprocessor.set_output(transform="pandas")

print("Pré-traitement défini.\n")

# ==============================================================================
# LIGHTGBM & SA PIPELINE
# ==============================================================================
print("\n" + "#"*30)
print("## DÉMARRAGE DU TEST: LightGBM ##")
print("#"*30)

# Activation de l'autologging MLflow
mlflow.sklearn.autolog(log_model_signatures=True, log_models=True)

# Modèle LightGBM
model_lgb = lgb.LGBMRegressor(random_state=42)

# Pipeline avec le preprocessor
full_pipeline_lgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', model_lgb)
])

# Hyperparamètres pour LightGBM
param_grid_lgb = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [3, 5, 7],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__num_leaves': [31, 50]
}

# GridSearchCV pour le modèle LightGBM
grid_search_lgb = GridSearchCV(
    estimator=full_pipeline_lgb,
    param_grid=param_grid_lgb,
    cv=3,
    n_jobs=-1,
    verbose=2,
    scoring='r2'
)

print("Modèle et pipeline prêts pour l'entraînement.\n")

# ==============================================================================
# EXPÉRIMENTATION MLFLOW POUR LIGHTGBM
# ==============================================================================
print("Lancement de l'expérimentation MLflow pour LightGBM...")
run_name_lgb = "Run_LightGBM_GridS_CV5" + datetime.now().strftime("%Y%m%d_%H%M%S")

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name_lgb) as run:
    # --- Entraînement ---
    grid_search_lgb.fit(X_train, Y_train)

    # --- Prédictions ---
    y_train_pred_lgb = grid_search_lgb.predict(X_train)
    y_test_pred_lgb = grid_search_lgb.predict(X_test)

    # --- Calcul des métriques ---
    r2_train_lgb = r2_score(Y_train, y_train_pred_lgb)
    mae_train_lgb = mean_absolute_error(Y_train, y_train_pred_lgb)
    rmse_train_lgb = np.sqrt(mean_squared_error(Y_train, y_train_pred_lgb))
    r2_test_lgb = r2_score(Y_test, y_test_pred_lgb)
    mae_test_lgb = mean_absolute_error(Y_test, y_test_pred_lgb)
    rmse_test_lgb = np.sqrt(mean_squared_error(Y_test, y_test_pred_lgb))

    cv_score_lgb = grid_search_lgb.best_score_

    # --- Logging MLflow (autolog se charge de la plupart des logs) ---
    print("\nEnregistrement des résultats dans MLflow...")

    # --- Sauvegarde du modèle (la pipeline complète) ---
    print("Modèle et métriques sauvegardés.")

    # ==============================================================================
    # AFFICHAGE DES RÉSULTATS POUR LIGHTGBM
    # ==============================================================================
    print("\n" + "="*60)
    print("🚀 LIGHTGBM - RÉSULTATS FINAUX 🚀")
    print(f"🎯 Run ID: {run.info.run_id}")
    print("="*60)

    print("\n🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:")
    for param, value in grid_search_lgb.best_params_.items():
        clean_param = param.replace('regressor__', '')
        print(f"  • {clean_param}: {value}")

    print(f"\n📊 MEILLEUR SCORE R² (Validation Croisée): {cv_score_lgb:.4f}")

    print("\n📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:")
    print(f"  • R² Score: {r2_train_lgb:.4f}")
    print(f"  • MAE:      {mae_train_lgb:.2f} $")
    print(f"  • RMSE:     {rmse_train_lgb:.2f} $")

    print("\n🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:")
    print(f"  • R² Score: {r2_test_lgb:.4f}")
    print(f"  • MAE:      {mae_test_lgb:.2f} $")
    print(f"  • RMSE:     {rmse_test_lgb:.2f} $")

    overfitting_r2_diff_lgb = r2_train_lgb - r2_test_lgb
    if overfitting_r2_diff_lgb > 0.1:
        print(f"\n⚠️  ATTENTION: Possible surapprentissage détecté (écart R²: {overfitting_r2_diff_lgb:.4f})")
    else:
        print(f"\n✅ Modèle bien généralisé (écart R²: {overfitting_r2_diff_lgb:.4f})")

    print("="*60)
    print(f"✅ Expérimentation terminée. Comparez les résultats dans MLflow UI.")
    print("="*60)


Pré-traitement défini.


##############################
## DÉMARRAGE DU TEST: LightGBM ##
##############################
Modèle et pipeline prêts pour l'entraînement.

Lancement de l'expérimentation MLflow pour LightGBM...


2025/08/22 17:11:06 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fitting 3 folds for each of 36 candidates, totalling 108 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002455 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002914 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002607 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002382 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Auto-choosing ro

2025/08/22 17:11:36 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).
"
2025/08/22 17:11:38 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot rep

🏃 View run powerful-goat-216 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/d4cdcb699a2b482abde3dd72fcb0985b
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run fearless-lynx-858 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/1720bdec1cd14fe2a0369d10d49d3d66
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run aged-sponge-829 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/63681e278004442eb6b85b355b259f77
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run illustrious-cub-401 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/bb6efde44f0c481cb27b2625bd71bed6
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run auspicious-ant-4 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/79f5f1f4f164489ab3db5e783017831c
🧪 View experiment at: https://ericjedha-getaroundml.hf

2025/08/22 17:11:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/08/22 17:11:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/


Enregistrement des résultats dans MLflow...
Modèle et métriques sauvegardés.

🚀 LIGHTGBM - RÉSULTATS FINAUX 🚀
🎯 Run ID: d5e98836691f42b3ae6d17a23bf20a16

🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:
  • learning_rate: 0.1
  • max_depth: 5
  • n_estimators: 300
  • num_leaves: 31

📊 MEILLEUR SCORE R² (Validation Croisée): 0.7195

📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:
  • R² Score: 0.8388
  • MAE:      8.32 $
  • RMSE:     12.87 $

🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:
  • R² Score: 0.7992
  • MAE:      10.19 $
  • RMSE:     14.53 $

✅ Modèle bien généralisé (écart R²: 0.0396)
✅ Expérimentation terminée. Comparez les résultats dans MLflow UI.
🏃 View run Run_LightGBM_GridS_CV520250822_171104 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/d5e98836691f42b3ae6d17a23bf20a16
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20


# Light GBM avec Regulation Ridge(L2)

In [31]:
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import mlflow
from datetime import datetime
import numpy as np

# ==============================================================================
# PREPROCESSING
# ==============================================================================
# Définition des colonnes numériques et catégorielles
numeric_features = ["mileage", "engine_power"]
categorical_features = ["model_key", "fuel", "paint_color", "car_type"]

# Création des transformateurs
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])

# Assemblage des transformateurs dans un ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="passthrough"
)
# Active la sortie en DataFrame pour conserver les noms de colonnes
preprocessor.set_output(transform="pandas")

print("Pré-traitement défini.\n")

# ==============================================================================
# LIGHTGBM & SA PIPELINE AVEC RÉGULARISATION RIDGE (L2)
# ==============================================================================
print("\n" + "#"*30)
print("## DÉMARRAGE DU TEST: LightGBM avec régularisation Ridge (L2) ##")
print("#"*30)

# Activation de l'autologging MLflow
mlflow.sklearn.autolog(log_model_signatures=True, log_models=True)

# Modèle LightGBM
model_lgb = lgb.LGBMRegressor(random_state=42)

# Pipeline avec le preprocessor
full_pipeline_lgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', model_lgb)
])

# Hyperparamètres pour LightGBM, incluant la régularisation Ridge (reg_lambda)
param_grid_lgb = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [3, 5],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__num_leaves': [31, 50],
    'regressor__reg_lambda': [0.0, 0.1, 1.0, 10.0],  # Régularisation Ridge (L2)
    'regressor__reg_alpha': [0.0]  # Désactive la régularisation L1 (Lasso)
}

# GridSearchCV pour le modèle LightGBM
grid_search_lgb = GridSearchCV(
    estimator=full_pipeline_lgb,
    param_grid=param_grid_lgb,
    cv=3,
    n_jobs=-1,
    verbose=2,
    scoring='r2'
)

print("Modèle et pipeline prêts pour l'entraînement.\n")

# ==============================================================================
# EXPÉRIMENTATION MLFLOW POUR LIGHTGBM
# ==============================================================================
print("Lancement de l'expérimentation MLflow pour LightGBM avec régularisation Ridge...")
run_name_lgb = "Run_LightGBM_Ridge_GridS_CV5" + datetime.now().strftime("%Y%m%d_%H%M%S")

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name_lgb) as run:
    # --- Entraînement ---
    grid_search_lgb.fit(X_train, Y_train)

    # --- Prédictions ---
    y_train_pred_lgb = grid_search_lgb.predict(X_train)
    y_test_pred_lgb = grid_search_lgb.predict(X_test)

    # --- Calcul des métriques ---
    r2_train_lgb = r2_score(Y_train, y_train_pred_lgb)
    mae_train_lgb = mean_absolute_error(Y_train, y_train_pred_lgb)
    rmse_train_lgb = np.sqrt(mean_squared_error(Y_train, y_train_pred_lgb))
    r2_test_lgb = r2_score(Y_test, y_test_pred_lgb)
    mae_test_lgb = mean_absolute_error(Y_test, y_test_pred_lgb)
    rmse_test_lgb = np.sqrt(mean_squared_error(Y_test, y_test_pred_lgb))

    cv_score_lgb = grid_search_lgb.best_score_

    # --- Logging MLflow (autolog se charge de la plupart des logs) ---
    print("\nEnregistrement des résultats dans MLflow...")

    # --- Sauvegarde du modèle (la pipeline complète) ---
    print("Modèle et métriques sauvegardés.")

    # ==============================================================================
    # AFFICHAGE DES RÉSULTATS POUR LIGHTGBM AVEC RÉGULARISATION RIDGE
    # ==============================================================================
    print("\n" + "="*60)
    print("🚀 LIGHTGBM AVEC RÉGULARISATION RIDGE - RÉSULTATS FINAUX 🚀")
    print(f"🎯 Run ID: {run.info.run_id}")
    print("="*60)

    print("\n🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:")
    for param, value in grid_search_lgb.best_params_.items():
        clean_param = param.replace('regressor__', '')
        print(f"  • {clean_param}: {value}")

    print(f"\n📊 MEILLEUR SCORE R² (Validation Croisée): {cv_score_lgb:.4f}")

    print("\n📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:")
    print(f"  • R² Score: {r2_train_lgb:.4f}")
    print(f"  • MAE:      {mae_train_lgb:.2f} $")
    print(f"  • RMSE:     {rmse_train_lgb:.2f} $")

    print("\n🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:")
    print(f"  • R² Score: {r2_test_lgb:.4f}")
    print(f"  • MAE:      {mae_test_lgb:.2f} $")
    print(f"  • RMSE:     {rmse_test_lgb:.2f} $")

    overfitting_r2_diff_lgb = r2_train_lgb - r2_test_lgb
    if overfitting_r2_diff_lgb > 0.1:
        print(f"\n⚠️  ATTENTION: Possible surapprentissage détecté (écart R²: {overfitting_r2_diff_lgb:.4f})")
    else:
        print(f"\n✅ Modèle bien généralisé (écart R²: {overfitting_r2_diff_lgb:.4f})")

    print("="*60)
    print(f"✅ Expérimentation terminée. Comparez les résultats dans MLflow UI.")
    print("="*60)


Pré-traitement défini.


##############################
## DÉMARRAGE DU TEST: LightGBM avec régularisation Ridge (L2) ##
##############################
Modèle et pipeline prêts pour l'entraînement.

Lancement de l'expérimentation MLflow pour LightGBM avec régularisation Ridge...


2025/08/22 17:14:24 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fitting 3 folds for each of 64 candidates, totalling 192 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002700 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001696 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 355
[LightGBM] [Info] Total Bins 360
[LightGBM] [Info] Number of data points in the train set: 2531, number of used features: 37
[LightGBM] [Info] Number of data points in the train set: 2531, number of used features: 35
[LightGBM] [Info] Start training from score 121.011853
[LightGBM] [Info] Start training from score 121.035559
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000752 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough,

2025/08/22 17:14:53 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).
"
2025/08/22 17:14:54 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot rep

🏃 View run legendary-boar-15 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/6259237c58084c60ab2e54c2129180e8
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run amusing-ray-547 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/3ae472f5ddfa4972b1c8de0e2c5ae472
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run ambitious-gnat-412 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/dec1088c6ba5410a94cd36003f95c1ca
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run vaunted-bass-471 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/aecc6daaa13344bb8658aea738642086
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run classy-bee-812 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/561c60db34da48609f81abf0ef2bda21
🧪 View experiment at: https://ericjedha-getaroundml.hf.spa


Enregistrement des résultats dans MLflow...
Modèle et métriques sauvegardés.

🚀 LIGHTGBM AVEC RÉGULARISATION RIDGE - RÉSULTATS FINAUX 🚀
🎯 Run ID: 57abbeeaea124f93a1bc9f51675203f4

🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:
  • learning_rate: 0.1
  • max_depth: 5
  • n_estimators: 200
  • num_leaves: 31
  • reg_alpha: 0.0
  • reg_lambda: 0.0

📊 MEILLEUR SCORE R² (Validation Croisée): 0.7185

📈 MÉTRIQUES SUR L'ENSEMBLE D'ENTRAÎNEMENT:
  • R² Score: 0.8216
  • MAE:      8.71 $
  • RMSE:     13.54 $

🎯 MÉTRIQUES SUR L'ENSEMBLE DE TEST:
  • R² Score: 0.7970
  • MAE:      10.28 $
  • RMSE:     14.61 $

✅ Modèle bien généralisé (écart R²: 0.0245)
✅ Expérimentation terminée. Comparez les résultats dans MLflow UI.
🏃 View run Run_LightGBM_Ridge_GridS_CV520250822_171423 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/57abbeeaea124f93a1bc9f51675203f4
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20


# LIGHTGBM + LASSO (L1) + AUTOMLOGGING + CORRECTION FEATURE NAMES

In [29]:
# ==============================================================================
# LIGHTGBM + LASSO (L1) + AUTOMLOGGING + CORRECTION FEATURE NAMES
# ==============================================================================
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from datetime import datetime
import numpy as np
import pandas as pd

# ⚙️ Activer l'autolog MLflow (après avoir configuré l'expériment)
mlflow.sklearn.autolog(log_model_signatures=True, log_models=True)

print("\n" + "#"*30)
print("## DÉMARRAGE DU TEST: LightGBM + Lasso (L1) ##")
print("#"*30)

# --- Préprocessing (amélioré avec feature_names_out) ---
numeric_features = ["mileage", "engine_power"]
categorical_features = ["model_key", 'fuel', "paint_color", "car_type"]

# 🔧 Ajouter feature_names_out pour éviter le warning
class SafeOneHotEncoder(OneHotEncoder):
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        return super().get_feature_names_out(input_features)

class SafeStandardScaler(StandardScaler):
    def get_feature_names_out(self, input_features=None):
        return input_features

# Création des transformateurs
numeric_transformer = Pipeline(steps=[
    ("scaler", SafeStandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", SafeOneHotEncoder(drop="first", handle_unknown="ignore"))
])

# Assemblage
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False  # Évite les préfixes 'num__' ou 'cat__' si tu veux des noms simples
)

# --- Modèle ---
model_lgb = lgb.LGBMRegressor(random_state=42, verbose=-1)

# Pipeline complète
full_pipeline_lgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', model_lgb)
])

# --- Grille d'hyperparamètres (réduite pour éviter 864 combinaisons) ---
param_grid_lgb = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [5, 7],
    'regressor__learning_rate': [0.1, 0.2],
    'regressor__num_leaves': [31, 63],
    'regressor__subsample': [0.8, 1.0],
    'regressor__colsample_bytree': [0.8, 1.0],
    'regressor__reg_alpha': [0.0, 0.1, 1.0, 5.0]  # 🔥 Lasso (L1 regularization)
}

# Réduction du CV si besoin
grid_search_lgb = GridSearchCV(
    estimator=full_pipeline_lgb,
    param_grid=param_grid_lgb,
    cv=3,
    n_jobs=-1,
    verbose=1,
    scoring='r2'
)

print("Modèle et pipeline prêts.\n")

# ==============================================================================
# 3. EXPÉRIMENTATION MLFLOW (avec autolog)
# ==============================================================================
print("Lancement de l'expérimentation MLflow avec autolog...")

run_name_lgb = "Run_LightGBM_Lasso_AutoLog_" + datetime.now().strftime("%Y%m%d_%H%M%S")

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name_lgb):
    # --- Entraînement (autolog enregistre tout) ---
    grid_search_lgb.fit(X_train, Y_train)

    # --- Prédictions ---
    y_train_pred = grid_search_lgb.predict(X_train)
    y_test_pred = grid_search_lgb.predict(X_test)

    # --- Métriques ---
    r2_train = r2_score(Y_train, y_train_pred)
    mae_train = mean_absolute_error(Y_train, y_train_pred)
    rmse_train = np.sqrt(mean_squared_error(Y_train, y_train_pred))

    r2_test = r2_score(Y_test, y_test_pred)
    mae_test = mean_absolute_error(Y_test, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(Y_test, y_test_pred))
    
    cv_score = grid_search_lgb.best_score_

    # 🔽 Pas besoin de log manuel : mlflow.sklearn.autolog() s'en occupe !
    # Mais on peut quand même log des métriques custom si besoin
    mlflow.log_metric("CV_score_R2", cv_score)
    mlflow.log_params(grid_search_lgb.best_params_)

    # --- Affichage ---
    print("\n" + "="*60)
    print("🚀 LIGHTGBM + LASSO (L1) - RÉSULTATS FINAUX 🚀")
    print(f"🎯 Run ID: {mlflow.active_run().info.run_id}")
    print("="*60)
    
    print("\n🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:")
    for param, value in grid_search_lgb.best_params_.items():
        clean_param = param.replace('regressor__', '')
        print(f"  • {clean_param}: {value}")
    
    print(f"\n📊 MEILLEUR SCORE R² (CV): {cv_score:.4f}")
    print(f"📈 R² Train: {r2_train:.4f} | R² Test: {r2_test:.4f}")
    print(f"📉 MAE Test: {mae_test:.2f} | RMSE Test: {rmse_test:.2f}")

    overfit = r2_train - r2_test
    print(f"📊 Écart R² (surapprentissage): {overfit:.4f}")
    print("="*60)


##############################
## DÉMARRAGE DU TEST: LightGBM + Lasso (L1) ##
##############################
Modèle et pipeline prêts.

Lancement de l'expérimentation MLflow avec autolog...


2025/08/22 21:28:29 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fitting 3 folds for each of 256 candidates, totalling 768 fits


/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarnin

🏃 View run unruly-gnat-868 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/085a044a0f1c437d8ca64ac5064fdafc
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run adventurous-bee-584 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/ef83b8d0539a44f2bd90a7d27cd6da4d
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run entertaining-rat-912 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/1a4bab838dc04fa889c7199659f09da3
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run delicate-kite-586 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/d2659b8977d44e51b30bcbf02c63a3a4
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20
🏃 View run sassy-kite-322 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/4c95a09ac3c54547b4b900e95dec9ac6
🧪 View experiment at: https://ericjedha-getaroundml.h

/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

2025/08/22 21:31:24 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/eric/Desktop/JEDHA/SMOLAGENT/.conda/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/la


🚀 LIGHTGBM + LASSO (L1) - RÉSULTATS FINAUX 🚀
🎯 Run ID: 42d459cea4a848feaae605fd0dcdd5d4

🔧 MEILLEURS HYPERPARAMÈTRES TROUVÉS:
  • colsample_bytree: 1.0
  • learning_rate: 0.1
  • max_depth: 5
  • n_estimators: 200
  • num_leaves: 31
  • reg_alpha: 5.0
  • subsample: 0.8

📊 MEILLEUR SCORE R² (CV): 0.7201
📈 R² Train: 0.8222 | R² Test: 0.7954
📉 MAE Test: 10.25 | RMSE Test: 14.67
📊 Écart R² (surapprentissage): 0.0268
🏃 View run Run_LightGBM_Lasso_AutoLog_20250822_212828 at: https://ericjedha-getaroundml.hf.space/#/experiments/20/runs/42d459cea4a848feaae605fd0dcdd5d4
🧪 View experiment at: https://ericjedha-getaroundml.hf.space/#/experiments/20


# Prediction avec le modèle sauvegardé grâce à MLFLOW

In [ ]:
import mlflow


# Set your variables for your environment
EXPERIMENT_NAME_1="Run_RandomForest_Regressor20250606_112905"

# Set tracking URI to your Hugging Face application
mlflow.set_tracking_uri(os.environ["APP_URI"])

# Set experiment's info 
mlflow.set_experiment(EXPERIMENT_NAME_1)

# Get our experiment info
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME_1)

logged_model = 'runs:/0593c0d718084afd833c75dd4845f895/best_estimator'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)

test = {
    "model_key": "Citroën",
    "mileage": 140411,
    "engine_power": 100,
    "fuel": "diesel",
    "paint_color": "red",
    "car_type": "convertible",
    "private_parking_available": 1,
    "has_gps": 1,
    "has_air_conditioning": 0,
    "automatic_car": 1,
    "has_getaround_connect": 1,
    "has_speed_regulator": 0,
    "winter_tires": 0
}
data = pd.DataFrame([test])



In [ ]:
data

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires
0,Citroën,140411,100,diesel,red,convertible,1,1,0,1,1,0,0


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
import pandas as pd

# Recréer le même preprocessor que lors de l'entraînement
numeric_features = ["mileage", "engine_power"]
categorical_features = ["model_key", "fuel", "paint_color", "car_type"]

numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="passthrough"
)
# Applique le preprocess avant l'inférence :
X_preprocessed = preprocessor.fit_transform(data)  # attention : fit_transform seulement si c’est test local
# En prod : utilise .transform() si déjà entraîné

X_dense = X_preprocessed.toarray() if hasattr(X_preprocessed, "toarray") else X_preprocessed
X_df = pd.DataFrame(X_dense, columns=preprocessor.get_feature_names_out())

In [ ]:

import mlflow

#reprendre le modèle depuis ML Flow et le faire tourner (copier coller, ne pas oublier de remettre les infos de l'expérience, Experiment

# Set your variables for your environment
EXPERIMENT_NAME_1="Run_FullPipeline_RF_20250606_134438"

# Set tracking URI to your Hugging Face application
mlflow.set_tracking_uri(os.environ["APP_URI"])

# Set experiment's info 
mlflow.set_experiment(EXPERIMENT_NAME_1)

# Get our experiment info
#experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME_1)

logged_model = 'runs:/f7019001dcb440f9b4a739550c670dba/best_estimator'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)


raw_data = pd.DataFrame([{
    "model_key": "Citroën",
    "mileage": 140000,
    "engine_power": 100,
    "fuel": "diesel",
    "paint_color": "black",
    "car_type": "convertible",
    "private_parking_available": 1,
    "has_gps": 1,
    "has_air_conditioning": 0,
    "automatic_car": 0,
    "has_getaround_connect": 1,
    "has_speed_regulator": 1,
    "winter_tires": 1
}])



preds = loaded_model.predict(raw_data)

response = {"prediction": preds.tolist()[0]}
response

{'prediction': 108.71319763460939}